# Lesson 6: Building AI Agents

In this lesson, you'll learn how to build AI agents that can plan, reason, and execute multi-step tasks.

## Topics Covered
1. Understanding AI agents and agent architectures
2. Building a basic agent with planning
3. Implementing agent memory systems
4. Multi-step task execution with ReAct pattern
5. Combining tools and RAG in agents

## Learning Objectives
- Understand what makes an AI agent different from simple function calling
- Implement the ReAct (Reasoning + Acting) pattern
- Build agents that can plan multi-step solutions
- Add memory systems to agents
- Combine tools and RAG for intelligent agents

In [ ]:
# Install required packages
#%pip install python-dotenv
#%pip install openai
#%pip install numpy

import os
from dotenv import load_dotenv
load_dotenv()
import openai
import numpy as np
import json
from typing import List, Dict, Optional, Callable, Any
from datetime import datetime
import time

print("✅ Packages loaded")
print(f"OpenAI version: {openai.__version__}")

In [ ]:
# Initialize OpenAI client
chat_client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=int(os.getenv("OPENAI_TIMEOUT", 30)),
    max_retries=int(os.getenv("MAX_RETRIES", 3)),
    base_url=os.getenv("OPENAI_ENDPOINT")
)

model_name = os.getenv("OPENAI_MODEL", "gpt-4.1")

print("✅ OpenAI client initialized")
print(f" --> Using model: {model_name}")

## 1. Understanding AI Agents

### What is an AI Agent?

An **AI Agent** is a system that can:
- **Perceive** its environment (receive input)
- **Think** (reason and plan)
- **Act** (use tools and take actions)
- **Learn** (remember and improve)

### Comparison:

| Feature | Simple Function Calling | AI Agent |
|---------|------------------------|----------|
| Planning | No | Yes |
| Multi-step | No | Yes |
| Memory | No | Yes |
| Self-correction | No | Yes |
| Tool chaining | Manual | Automatic |

### Agent Architecture Analogy:

**Function Calling** = Following a recipe step-by-step
**AI Agent** = A chef who can:
- Plan the meal based on available ingredients
- Decide which tools to use and when
- Adjust the plan if something goes wrong
- Remember what worked before

In [ ]:
# Example 1A: Simple vs Agent approach
print("=" * 80)
print("COMPARISON: Function Calling vs Agent")
print("=" * 80)

task = "Book a restaurant for 4 people tomorrow at 7pm, then send calendar invite to the team"

print(f"\n📋 Task: {task}\n")

print("❌ Simple Function Calling:")
print("   - You must manually break down the task")
print("   - You must call: book_restaurant() then send_calendar_invite()")
print("   - If booking fails, you must handle it manually")
print("   - No memory of previous bookings\n")

print("✅ AI Agent:")
print("   - Agent understands it needs 2 steps")
print("   - Agent plans: First book, then send invite")
print("   - Agent can retry or find alternatives if booking fails")
print("   - Agent remembers team preferences from past bookings")

print("\n" + "="*80)
print("Key Insight: Agents can PLAN and ADAPT, not just execute")
print("="*80)

## 2. Building a Basic Agent with Planning

### Core Agent Loop:

```
1. THINK: Analyze the task and plan
2. ACT: Execute an action (use a tool)
3. OBSERVE: Check the result
4. REPEAT: Until task is complete
```

This is called the **Think-Act-Observe loop**.

In [ ]:
# Example 2A: Basic Agent with Planning

class BasicAgent:
    """Simple agent that can plan and execute tasks"""
    
    def __init__(self, tools: Dict[str, Callable], max_iterations: int = 5):
        self.tools = tools
        self.max_iterations = max_iterations
        self.execution_log = []
    
    def run(self, task: str) -> Dict:
        """Execute a task using planning and tools"""
        print(f"\n🎯 Task: {task}\n")
        print("=" * 80)
        
        # Create initial plan
        plan = self._create_plan(task)
        print(f"\n📋 Plan created:\n{plan}\n")
        print("=" * 80)
        
        # Execute the plan
        for iteration in range(self.max_iterations):
            print(f"\n🔄 Iteration {iteration + 1}/{self.max_iterations}")
            print("─" * 80)
            
            # Decide next action
            action = self._decide_next_action(task, plan)
            
            if action['type'] == 'COMPLETE':
                print("\n✅ Task completed!")
                return {
                    'status': 'success',
                    'result': action['result'],
                    'iterations': iteration + 1,
                    'log': self.execution_log
                }
            
            # Execute action
            print(f"\n💭 Thinking: {action['reasoning']}")
            print(f"🔧 Action: {action['tool']} with {action['args']}")
            
            result = self._execute_action(action)
            print(f"📊 Result: {result}")
            
            # Log execution
            self.execution_log.append({
                'iteration': iteration + 1,
                'action': action,
                'result': result
            })
        
        return {
            'status': 'max_iterations_reached',
            'log': self.execution_log
        }
    
    def _create_plan(self, task: str) -> str:
        """Create an execution plan for the task"""
        tools_desc = "\n".join([f"- {name}: {func.__doc__}" for name, func in self.tools.items()])
        
        prompt = f"""You are an AI planning assistant. Create a step-by-step plan for this task.

Available tools:
{tools_desc}

Task: {task}

Create a clear, numbered plan (3-5 steps max):"""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            # max_tokens=32000
        )
        
        return response.choices[0].message.content
    
    def _decide_next_action(self, task: str, plan: str) -> Dict:
        """Decide what action to take next"""
        # Build context from execution log
        history = "\n".join([
            f"Step {log['iteration']}: Used {log['action']['tool']} → {log['result']}"
            for log in self.execution_log
        ])
        
        tools_desc = json.dumps([{"name": name, "description": func.__doc__} 
                                 for name, func in self.tools.items()], indent=2)
        
        prompt = f"""You are an AI agent deciding the next action.

Task: {task}
Plan: {plan}

Execution history:
{history if history else 'No actions taken yet'}

Available tools:
{tools_desc}

Decide the next action. Return JSON in this format:
{{
  "reasoning": "why this action",
  "type": "ACTION" or "COMPLETE",
  "tool": "tool_name" (if ACTION),
  "args": {{}} (if ACTION),
  "result": "final answer" (if COMPLETE)
}}

Return ONLY the JSON, no other text."""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            # max_tokens=32000,
            response_format={"type": "json_object"}
        )
        
        return json.loads(response.choices[0].message.content)
    
    def _execute_action(self, action: Dict) -> Any:
        """Execute a tool action"""
        tool_name = action['tool']
        args = action['args']
        
        if tool_name not in self.tools:
            return f"Error: Tool '{tool_name}' not found"
        
        try:
            return self.tools[tool_name](**args)
        except Exception as e:
            return f"Error executing {tool_name}: {str(e)}"

print("✅ BasicAgent class created")

In [ ]:
# Example 2B: Define tools for the agent

# def calculate(operation: str, a: float, b: float) -> float:
#     """Perform mathematical operations: add, subtract, multiply, divide"""
#     ops = {
#         'add': a + b,
#         'subtract': a - b,
#         'multiply': a * b,
#         'divide': a / b if b != 0 else "Error: Division by zero"
#     }
#     return ops.get(operation, "Unknown operation")

def calculate(operation: str = None, a: float = None, b: float = None, expression: str = None) -> float:
    """Perform mathematical operations: add, subtract, multiply, divide
    
    Use one of two methods:
    1. Simple: calculate(operation='add', a=10, b=5)
    2. Expression: calculate(expression='10 + 5')
    """
    # Handle expression-based calls
    if expression is not None:
        try:
            # Safely evaluate simple math expressions
            allowed_chars = set('0123456789+-*/(). ')
            if all(c in allowed_chars for c in expression):
                return eval(expression)
        except:
            return "Error: Invalid expression"
    
    # Handle operation-based calls
    if a is not None and b is not None:
        ops = {
            'add': a + b,
            'subtract': a - b,
            'multiply': a * b,
            'divide': a / b if b != 0 else "Error: Division by zero"
        }
        return ops.get(operation, "Unknown operation")
    
    return "Error: Provide either (operation, a, b) or expression parameter"

def get_current_date() -> str:
    """Get the current date in YYYY-MM-DD format"""
    return datetime.now().strftime("%Y-%m-%d")

def search_database(query: str) -> Dict:
    """Search a product database (simulated)"""
    # Simulated database
    products = {
        'laptop': {'name': 'ProBook 15', 'price': 899, 'stock': 5},
        'mouse': {'name': 'Wireless Mouse', 'price': 25, 'stock': 50},
        'keyboard': {'name': 'Mechanical Keyboard', 'price': 75, 'stock': 20}
    }
    
    for key, product in products.items():
        if key in query.lower():
            return product
    return {'error': 'Product not found'}

def send_email(to: str, subject: str, body: str) -> str:
    """Send an email (simulated)"""
    return f"Email sent to {to}: '{subject}'"

# Create tool registry
agent_tools = {
    'calculate': calculate,
    #'calculate': calculate_advanced,
    'get_current_date': get_current_date,
    'search_database': search_database,
    'send_email': send_email
}

print("✅ Agent tools defined:")
for name, func in agent_tools.items():
    print(f"   • {name}: {func.__doc__}")

In [ ]:
# Example 2C: Test the basic agent
agent = BasicAgent(agent_tools, max_iterations=5)

print("\n" + "=" * 80)
print("TESTING BASIC AGENT")
print("=" * 80)

# Test with a multi-step task
task = "Find the price of a laptop in the database, calculate 10% discount, and tell me the final price"

result = agent.run(task)

print("\n" + "=" * 80)
print(f"\n📊 Result: {result['status']}")
print(f"🔢 Iterations used: {result.get('iterations', 'N/A')}")
if result.get('result'):
    print(f"✨ Final Answer: {result['result']}")

## 3. Agent Memory Systems

Agents need memory to:
- Remember previous interactions
- Learn from past successes/failures
- Maintain context across tasks

### Types of Memory:

1. **Short-term Memory**: Current conversation/task
2. **Long-term Memory**: Persistent knowledge
3. **Working Memory**: Active reasoning and planning

**Analogy:** Like a human:
- Short-term = What we're talking about right now
- Long-term = Things we learned years ago
- Working = Information we're actively processing

In [ ]:
# Example 3A: Agent with Memory

class MemoryAgent:
    """Agent with short-term and long-term memory"""
    
    def __init__(self, tools: Dict[str, Callable]):
        self.tools = tools
        
        # Short-term memory: Current conversation
        self.short_term_memory = []
        
        # Long-term memory: Persistent facts and learnings
        self.long_term_memory = [
            "User prefers concise answers",
            "Previous successful approach: break complex tasks into steps"
        ]
        
        # Working memory: Current task state
        self.working_memory = {}
        
        self.max_iterations = 5
    
    def run(self, task: str, remember: bool = True) -> Dict:
        """Execute task with memory awareness"""
        print(f"\n🎯 Task: {task}\n")
        
        # Initialize working memory for this task
        self.working_memory = {
            'task': task,
            'start_time': datetime.now(),
            'steps_completed': []
        }
        
        # Add to short-term memory
        if remember:
            self.short_term_memory.append({
                'timestamp': datetime.now().isoformat(),
                'task': task
            })
        
        print("🧠 Memory Status:")
        print(f"   Short-term: {len(self.short_term_memory)} items")
        print(f"   Long-term: {len(self.long_term_memory)} learnings")
        if self.short_term_memory:
            print(f"   Recent context: {self.short_term_memory[-1]['task'] if self.short_term_memory else 'None'}")
        print()
        
        # Execute with memory awareness
        for iteration in range(self.max_iterations):
            action = self._decide_action_with_memory(task)
            
            if action['type'] == 'COMPLETE':
                # Learn from this experience
                self._update_long_term_memory(task, action['result'])
                
                return {
                    'status': 'success',
                    'result': action['result'],
                    'iterations': iteration + 1
                }
            
            # Execute and remember
            print(f"\n[Step {iteration + 1}] {action['tool']}: {action['reasoning']}")
            result = self.tools[action['tool']](**action['args'])
            print(f"   → {result}")
            
            self.working_memory['steps_completed'].append({
                'step': iteration + 1,
                'action': action['tool'],
                'result': result
            })
        
        return {'status': 'max_iterations', 'memory': self.working_memory}
    
    def _decide_action_with_memory(self, task: str) -> Dict:
        """Decide next action using all memory types"""
        # Build context from memories
        context = f"""Task: {task}

Long-term learnings:
{chr(10).join([f'- {item}' for item in self.long_term_memory])}

Recent context:
{chr(10).join([f'- {m["task"]}' for m in self.short_term_memory[-3:]]) if self.short_term_memory else 'None'}

Current progress:
{json.dumps(self.working_memory['steps_completed'], indent=2) if self.working_memory['steps_completed'] else 'Just started'}

Available tools: {list(self.tools.keys())}

Decide next action as JSON:
{{
  "reasoning": "why",
  "type": "ACTION" or "COMPLETE",
  "tool": "name" (if ACTION),
  "args": {{}} (if ACTION),
  "result": "answer" (if COMPLETE)
}}"""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": context}],
            temperature=0.3,
            # max_tokens=32000,
            response_format={"type": "json_object"}
        )
        
        return json.loads(response.choices[0].message.content)
    
    def _update_long_term_memory(self, task: str, result: str):
        """Extract learnings and update long-term memory"""
        # Simple heuristic: remember successful patterns
        if len(self.working_memory['steps_completed']) > 0:
            pattern = f"For '{task[:30]}...' tasks: use {len(self.working_memory['steps_completed'])} steps"
            if pattern not in self.long_term_memory:
                self.long_term_memory.append(pattern)
                print(f"\n📚 Learned: {pattern}")
    
    def recall(self, query: str) -> List[str]:
        """Search memories for relevant information"""
        relevant = []
        query_lower = query.lower()
        
        # Search long-term memory
        for memory in self.long_term_memory:
            if any(word in memory.lower() for word in query_lower.split()):
                relevant.append(f"[Long-term] {memory}")
        
        # Search short-term memory
        for memory in self.short_term_memory[-5:]:
            if any(word in memory['task'].lower() for word in query_lower.split()):
                relevant.append(f"[Short-term] {memory['task']}")
        
        return relevant
    
    def clear_short_term(self):
        """Clear short-term memory"""
        self.short_term_memory = []
        print("🧹 Short-term memory cleared")

print("✅ MemoryAgent class created")

In [ ]:
# Example 3B: Test agent with memory
memory_agent = MemoryAgent(agent_tools)

print("=" * 80)
print("TESTING AGENT WITH MEMORY")
print("=" * 80)

# Task 1
print("\n📍 TASK 1")
result1 = memory_agent.run("What's today's date?")

# Task 2 - agent should remember context
print("\n\n📍 TASK 2")
result2 = memory_agent.run("Find laptop price and calculate total price with 15% tax")

# Show memory
print("\n" + "=" * 80)
print("\n🧠 Agent's Memory:")
print(f"\n📝 Short-term memory ({len(memory_agent.short_term_memory)} items):")
for i, mem in enumerate(memory_agent.short_term_memory, 1):
    print(f"   {i}. {mem['task']}")

print(f"\n📚 Long-term memory ({len(memory_agent.long_term_memory)} learnings):")
for i, learning in enumerate(memory_agent.long_term_memory, 1):
    print(f"   {i}. {learning}")

# Test recall
print("\n🔍 Testing memory recall:")
query = "laptop price"
recalled = memory_agent.recall(query)
print(f"   Query: '{query}'")
print(f"   Found {len(recalled)} relevant memories:")
for mem in recalled:
    print(f"     - {mem}")

## 4. ReAct Pattern: Reasoning + Acting

**ReAct** is a powerful agent pattern that interleaves:
- **Reasoning**: Thinking about what to do
- **Acting**: Executing actions with tools

### ReAct Loop:
```
1. Thought: "I need to find the price"
2. Action: search_database("laptop")
3. Observation: {price: 899}
4. Thought: "Now I need to calculate discount"
5. Action: calculate("multiply", 899, 0.9)
6. Observation: 809.1
7. Thought: "Task complete"
8. Answer: "The discounted price is $809.10"
```

**Why ReAct works:** The agent explicitly reasons before each action, making its decision process transparent and more reliable.

In [ ]:
# Example 4A: ReAct Agent Implementation

class ReActAgent:
    """Agent using the ReAct (Reasoning + Acting) pattern"""
    
    def __init__(self, tools: Dict[str, Callable], max_iterations: int = 10):
        self.tools = tools
        self.max_iterations = max_iterations
        self.trace = []  # Store thought-action-observation sequence
    
    def run(self, task: str, verbose: bool = True) -> Dict:
        """Execute task using ReAct pattern"""
        if verbose:
            print(f"\n🎯 Task: {task}\n")
            print("=" * 80)
            print("ReAct Loop (Thought → Action → Observation)")
            print("=" * 80)
        
        self.trace = []
        
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n🔄 Iteration {iteration + 1}")
                print("─" * 80)
            
            # THINK: Generate reasoning
            thought = self._generate_thought(task)
            if verbose:
                print(f"\n💭 Thought: {thought['reasoning']}")
            
            # Check if task is complete
            if thought['status'] == 'COMPLETE':
                if verbose:
                    print(f"\n✅ Final Answer: {thought['answer']}")
                    print("\n" + "=" * 80)
                    self._print_trace()
                
                return {
                    'status': 'success',
                    'answer': thought['answer'],
                    'iterations': iteration + 1,
                    'trace': self.trace
                }
            
            # ACT: Execute the planned action
            action = thought['action']
            if verbose:
                print(f"🔧 Action: {action['tool']}({action['args']})")
            
            observation = self._execute_tool(action['tool'], action['args'])
            if verbose:
                print(f"👀 Observation: {observation}")
            
            # RECORD: Add to trace
            self.trace.append({
                'thought': thought['reasoning'],
                'action': f"{action['tool']}({action['args']})",
                'observation': observation
            })
        
        return {
            'status': 'max_iterations',
            'trace': self.trace
        }
    
    def _generate_thought(self, task: str) -> Dict:
        """Generate reasoning and decide next action"""
        # Build context from trace
        trace_text = "\n".join([
            f"Thought: {t['thought']}\nAction: {t['action']}\nObservation: {t['observation']}"
            for t in self.trace
        ])
        
        tools_json = json.dumps([
            {"name": name, "description": func.__doc__}
            for name, func in self.tools.items()
        ], indent=2)
        
        prompt = f"""You are a ReAct agent. Think step-by-step and decide the next action.

Task: {task}

Available tools:
{tools_json}

Trace so far:
{trace_text if trace_text else 'Just started'}

Think carefully and respond with JSON:
{{
  "reasoning": "Your step-by-step thinking",
  "status": "CONTINUE" or "COMPLETE",
  "action": {{
    "tool": "tool_name",
    "args": {{}}
  }} (if CONTINUE),
  "answer": "final answer" (if COMPLETE)
}}

Remember: Think before acting!"""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        
        return json.loads(response.choices[0].message.content)
    
    def _execute_tool(self, tool_name: str, args: Dict) -> Any:
        """Execute a tool and return observation"""
        if tool_name not in self.tools:
            return f"Error: Tool '{tool_name}' not found"
        
        try:
            return self.tools[tool_name](**args)
        except Exception as e:
            return f"Error: {str(e)}"
    
    def _print_trace(self):
        """Print the complete thought-action-observation trace"""
        print("\n📋 Complete ReAct Trace:")
        print("=" * 80)
        for i, step in enumerate(self.trace, 1):
            print(f"\nStep {i}:")
            print(f"  💭 Thought: {step['thought']}")
            print(f"  🔧 Action: {step['action']}")
            print(f"  👀 Observation: {step['observation']}")
        print("\n" + "=" * 80)

print("✅ ReActAgent class created")

In [ ]:
# Example 4B: Test ReAct Agent
react_agent = ReActAgent(agent_tools, max_iterations=10)

print("=" * 80)
print("TESTING ReAct AGENT")
print("=" * 80)

# Complex multi-step task
task = """Find the laptop price, apply a 20% discount, 
calculate 8% sales tax on the discounted price, 
and tell me the final amount to pay."""

result = react_agent.run(task, verbose=True)

print(f"\n\n📊 Summary:")
print(f"   Status: {result['status']}")
print(f"   Iterations: {result.get('iterations', 'N/A')}")
print(f"   Steps in trace: {len(result['trace'])}")

## 5. Combining Tools and RAG in Agents

The most powerful agents combine:
- **Tools** for actions (from Lesson 4)
- **RAG** for knowledge (from Lesson 5)
- **Planning** for multi-step tasks
- **Memory** for context

**Use case example:**
"Research our competitors' pricing, compare with our products, and send a summary email to the team"

This requires:
1. RAG to search company docs
2. Tools to fetch competitor data
3. Tools to send email
4. Planning to orchestrate it all

In [ ]:
# Example 5A: Helper functions for RAG
def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding for text"""
    text = text.replace("\n", " ")
    response = chat_client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity"""
    vec1, vec2 = np.array(vec1), np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

class SimpleKnowledgeBase:
    """Simple RAG knowledge base"""
    
    def __init__(self):
        self.documents = []
        self.embeddings = []
    
    def add_document(self, text: str):
        """Add a document to knowledge base"""
        self.documents.append(text)
        self.embeddings.append(get_embedding(text))
    
    def search(self, query: str, top_k: int = 2) -> List[str]:
        """Search for relevant documents"""
        query_embedding = get_embedding(query)
        
        similarities = [
            (i, cosine_similarity(query_embedding, emb))
            for i, emb in enumerate(self.embeddings)
        ]
        
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        return [self.documents[i] for i, _ in similarities[:top_k]]

print("✅ RAG helper functions ready")

In [ ]:
# Example 5B: Hybrid Agent (Tools + RAG)

class HybridAgent:
    """Agent that combines tools and knowledge base"""
    
    def __init__(self, tools: Dict[str, Callable], knowledge_base: SimpleKnowledgeBase):
        self.tools = tools
        self.kb = knowledge_base
        self.max_iterations = 10
    
    def run(self, task: str) -> Dict:
        """Execute task using both tools and knowledge"""
        print(f"\n🎯 Task: {task}\n")
        print("=" * 80)
        
        for iteration in range(self.max_iterations):
            print(f"\n🔄 Step {iteration + 1}")
            
            # Decide: Tool or Knowledge search?
            decision = self._decide_approach(task, iteration)
            
            print(f"💭 Decision: {decision['reasoning']}")
            
            if decision['type'] == 'COMPLETE':
                print(f"\n✅ Answer: {decision['answer']}")
                return {
                    'status': 'success',
                    'answer': decision['answer'],
                    'iterations': iteration + 1
                }
            
            elif decision['type'] == 'SEARCH_KNOWLEDGE':
                query = decision['query']
                print(f"📚 Searching knowledge: '{query}'")
                docs = self.kb.search(query, top_k=2)
                print(f"   Found {len(docs)} relevant documents")
                for i, doc in enumerate(docs, 1):
                    print(f"   {i}. {doc[:80]}...")
            
            elif decision['type'] == 'USE_TOOL':
                tool = decision['tool']
                args = decision['args']
                print(f"🔧 Using tool: {tool}({args})")
                result = self.tools[tool](**args)
                print(f"   Result: {result}")
        
        return {'status': 'max_iterations'}
    
    def _decide_approach(self, task: str, iteration: int) -> Dict:
        """Decide whether to use tools or search knowledge"""
        tools_desc = json.dumps([{"name": n, "desc": f.__doc__} 
                                 for n, f in self.tools.items()], indent=2)
        
        prompt = f"""You are a hybrid agent with tools AND a knowledge base.

Task: {task}
Step: {iteration + 1}

Available tools:
{tools_desc}

Knowledge base: Company policies, product info, FAQs

Decide next step as JSON:
{{
  "reasoning": "why this approach",
  "type": "SEARCH_KNOWLEDGE" or "USE_TOOL" or "COMPLETE",
  "query": "search query" (if SEARCH_KNOWLEDGE),
  "tool": "tool_name" (if USE_TOOL),
  "args": {{}} (if USE_TOOL),
  "answer": "final answer" (if COMPLETE)
}}

Guidelines:
- Return ONLY one JSON object and nothing else
- Use SEARCH_KNOWLEDGE for: policies, documentation, facts
- Use USE_TOOL for: calculations, current data, actions
- Use COMPLETE when task is done"""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-5.2"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        
        return json.loads(response.choices[0].message.content)

print("✅ HybridAgent class created")

In [ ]:
# Example 5C: Test Hybrid Agent
knowledge_base = SimpleKnowledgeBase()

# Add sample documents
knowledge_base.add_document("Our return policy allows returns within 30 days of purchase with a receipt.")
knowledge_base.add_document("The ProBook 15 laptop is priced at $899 and comes with a 1-year warranty.")
knowledge_base.add_document("We offer free shipping on orders over $50 within the continental US.")

hybrid_agent = HybridAgent(agent_tools, knowledge_base)

print("=" * 80)
print("TESTING HYBRID AGENT")
print("=" * 80)

# Task
task = """Find the price of the ProBook 15 laptop, 
check our return policy, and send an email summary to the sales team."""

result = hybrid_agent.run(task)

print(f"\n\n📊 Summary:")
print(f"   Status: {result['status']}")
print(f"   Iterations: {result.get('iterations', 'N/A')}")